# 🎬 Supernan Hindi Dubbing Pipeline

**Zero-cost pipeline: English video → Hindi dubbed video with lip sync**

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

Pipeline stages:
1. Install dependencies
2. Clone VideoReTalking & GFPGAN
3. Upload & extract 15-second clip
4. Whisper transcription
5. IndicTrans2 Hindi translation
6. Coqui XTTS v2 voice cloning
7. Audio speed adjustment
8. VideoReTalking lip-sync
9. GFPGAN face enhancement
10. Download output

In [1]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import subprocess, torch
try:
    r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(r.stdout if r.returncode == 0 else 'No NVIDIA GPU.')
except FileNotFoundError:
    print('No nvidia-smi (Mac/CPU mode).')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')


No nvidia-smi (Mac/CPU mode).
Device: cpu


In [2]:
# ── Cell 2: Install System Deps ───────────────────────────────────────────────
!apt-get install -y ffmpeg wget unzip -q
print('✓ ffmpeg installed')

zsh(68308) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68308) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh:1: command not found: apt-get
✓ ffmpeg installed


In [3]:
# ── Cell 3: Install Python packages ──────────────────────────────────────────
# PyTorch with CUDA (Colab default, but ensuring compatibility)
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

# Core pipeline deps
!pip install -q openai-whisper TTS transformers sentencepiece sacremoses
!pip install -q git+https://github.com/VarunGumma/IndicTransTokenizer.git
!pip install -q pydub librosa soundfile deep-translator
!pip install -q basicsr facexlib gfpgan realesrgan
print("✓ All packages installed")


zsh(68313) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68313) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
python3.11(68313) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
Python(68313) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(68316) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68316) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
python3.11(68316) MallocStackLogging: could not tag MSL-re

In [4]:
# ── Cell 4: Clone Repos ───────────────────────────────────────────────────────
import os

# VideoReTalking
if not os.path.exists('VideoReTalking'):
    !git clone -q https://github.com/vinthony/video-retalking VideoReTalking
    %pip install -q -r VideoReTalking/requirements.txt
print('✓ VideoReTalking ready')

# GFPGAN
if not os.path.exists('GFPGAN'):
    !git clone -q https://github.com/TencentARC/GFPGAN GFPGAN
    %pip install -q basicsr facexlib gfpgan
    %cd GFPGAN
    !python setup.py develop -q
    %cd ..
print('✓ GFPGAN ready')

✓ VideoReTalking ready
✓ GFPGAN ready


In [5]:
# ── Cell 5: Download VideoReTalking Weights ──────────────────────────────────
import os
os.makedirs("models/VideoReTalking", exist_ok=True)
!wget -q https://github.com/vinthony/video-retalking/releases/download/v0.0.1/30_net_G.pth -O models/VideoReTalking/30_net_G.pth
!wget -q https://github.com/vinthony/video-retalking/releases/download/v0.0.1/BFM.zip -O models/VideoReTalking/BFM.zip
!unzip -qo models/VideoReTalking/BFM.zip -d models/VideoReTalking/
print("✓ VideoReTalking weights ready")


zsh(68344) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68344) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
wget(68344) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(68349) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68349) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
wget(68349) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
^C
zsh(68480) MallocStackLogging: process is not in a debuggable e

In [6]:
# ── Cell 6: Download GFPGAN Weights ──────────────────────────────────────────
os.makedirs('GFPGAN/experiments/pretrained_models', exist_ok=True)
gfpgan_weight = 'GFPGAN/experiments/pretrained_models/GFPGANv1.4.pth'
if not os.path.exists(gfpgan_weight):
    !wget -q --show-progress -O {gfpgan_weight} https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth
print('✓ GFPGAN weights ready')

✓ GFPGAN weights ready


In [7]:
# ── Cell 7: Download Input Video from Google Drive ──────────────────────────
import os, subprocess

# ── Paste your Google Drive share link here ───────────────────────────────────
GDRIVE_URL = 'https://drive.google.com/file/d/1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW/view?usp=sharing'

INPUT_VIDEO = 'supernan_training.mp4'

if not os.path.exists(INPUT_VIDEO):
    print('Downloading video from Google Drive...')
    # Install gdown if needed
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True)
    import gdown
    gdown.download(url=GDRIVE_URL, output=INPUT_VIDEO, fuzzy=True)

if os.path.exists(INPUT_VIDEO):
    size_mb = os.path.getsize(INPUT_VIDEO) / 1e6
    print(f'✓ Video ready: {INPUT_VIDEO} ({size_mb:.1f} MB)')
else:
    print('⚠️  Download failed. If the file is restricted, share it as "Anyone with the link"')
    print('   Or place the video manually in:', os.getcwd())


✓ Video ready: supernan_training.mp4 (621.9 MB)


In [8]:
# ── Cell 8: Configuration ────────────────────────────────────────────────────
# ⬇️ Change these if you want a different segment
START_SEC = 30
END_SEC   = 50

os.makedirs('workspace', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('models', exist_ok=True)

print(f'Processing segment: {START_SEC}s – {END_SEC}s ({END_SEC - START_SEC}s clip)')

Processing segment: 30s – 50s (20s clip)


In [9]:
# ── Cell 9: Stage 1 — Extract Clip ───────────────────────────────────────────
CLIP = 'workspace/clip.mp4'
duration = END_SEC - START_SEC

!ffmpeg -y -ss {START_SEC} -i "{INPUT_VIDEO}" -t {duration} \
    -c:v libx264 -c:a aac -preset fast -crf 18 {CLIP} -loglevel warning

from IPython.display import Video
print('✓ Clip extracted')
Video(CLIP, width=640)

zsh(68507) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68507) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(68507) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
✓ Clip extracted


In [10]:
# ── Cell 10: Stage 2 — Extract Audio ─────────────────────────────────────────
AUDIO_16K = 'workspace/clip_audio_16k.wav'
AUDIO_REF  = 'workspace/clip_audio_ref22k.wav'

# 16kHz mono for Whisper STT
!ffmpeg -y -i {CLIP} -vn -acodec pcm_s16le -ar 16000 -ac 1 {AUDIO_16K} -loglevel warning

# 22050 Hz for XTTS v2 — denoise + highpass (remove rumble) + normalize
# afftdn: spectral noise reduction | highpass: remove <80Hz rumble
# loudnorm: consistent volume for voice cloning
!ffmpeg -y -i {CLIP} -vn -ar 22050 -ac 1 \
    -af "afftdn=nf=-25,highpass=f=80,lowpass=f=8000,loudnorm=I=-16:LRA=11:TP=-1.5" \
    {AUDIO_REF} -loglevel warning

import subprocess, os
dur = float(subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration',
    '-of','csv=p=0', AUDIO_REF], capture_output=True, text=True).stdout.strip())
print(f'✓ Audio extracted: 16kHz (Whisper) + 22kHz denoised reference ({dur:.1f}s)')


zsh(68533) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68533) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(68533) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(68534) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68534) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(68534) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
✓ Audio extracted: 16kHz (Whisper) + 22kHz denoised reference 

In [11]:
# ── Cell 11: Stage 3 — Transcribe with Whisper ───────────────────────────────
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import os
if "AUDIO_16K" not in globals(): AUDIO_16K = "workspace/clip_audio_16k.wav"
import whisper

print('Loading Whisper medium model (downloading ~1.5 GB on first run)...')
model = whisper.load_model('medium')  # 'small' is faster; 'large' is more accurate

result = model.transcribe(AUDIO_16K, language='en', word_timestamps=True, verbose=False)
ENGLISH_TEXT = result['text'].strip()

with open('workspace/transcript_en.txt', 'w') as f:
    f.write(ENGLISH_TEXT)

print(f'✓ Transcript:\n{ENGLISH_TEXT}')

Loading Whisper medium model (downloading ~1.5 GB on first run)...


/Users/vikashkumar/Desktop/Supernan/venv/lib/python3.11/site-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
100%|██████████| 2001/2001 [00:24<00:00, 80.57frames/s]

✓ Transcript:
Clean your body, take bath, put deodorant, put a cloth Why do you do all these? Because you are always around your child Your child is always around you So if you smell your child's body or smell their breath Children will not like it They will not come to you


In [12]:
# ── Cell 12: Stage 4 — Translate to Hindi (IndicTrans2) ──────────────────────
import torch, os
if "DEVICE" not in globals(): DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if "ENGLISH_TEXT" not in globals():
    if os.path.exists("workspace/transcript_en.txt"):
        with open("workspace/transcript_en.txt", "r") as f: ENGLISH_TEXT = f.read().strip()
        print("✓ ENGLISH_TEXT recovered from disk")
    else: raise NameError("ENGLISH_TEXT missing. Run Cell 11.")

def translate_indictrans2(text):
    try:
        from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
        from IndicTransToolkit import IndicProcessor
        MODEL = "ai4bharat/indictrans2-en-indic-1B"
        print("Loading IndicTrans2...")
        tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
        model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, trust_remote_code=True).to(DEVICE)
        ip = IndicProcessor(inference=True)
        sentences = [s.strip() for s in text.split(".") if s.strip()]
        batch = ip.preprocess_batch(sentences, src_lang="eng_Latn", tgt_lang="hin_Deva")
        inputs = tokenizer(batch, truncation=True, padding="longest", return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            outputs = model.generate(**inputs, num_beams=5, max_length=256)
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        postprocessed = ip.postprocess_batch(decoded, lang="hin_Deva")
        return " ".join(postprocessed)
    except Exception as e:
        print(f"IndicTrans2 failed: {e}")
        return None

print("Attempting Hindi translation...")
HINDI_TEXT = translate_indictrans2(ENGLISH_TEXT)

if not HINDI_TEXT:
    print("⚠️ IndicTrans2 failed. Attempting Google Translate fallback...")
    try:
        from deep_translator import GoogleTranslator
        HINDI_TEXT = GoogleTranslator(source="en", target="hi").translate(ENGLISH_TEXT)
        print("✓ Google Translate success!")
    except Exception as ge:
        print(f"❌ Google Translate failed: {ge}. Using English as last resort.")
        HINDI_TEXT = ENGLISH_TEXT

with open("workspace/translation_hi.txt", "w", encoding="utf-8") as f: f.write(HINDI_TEXT)
print(f"✓ Hindi translation completed.")
print(f"Text: {HINDI_TEXT[:100]}...")


Attempting Hindi translation...
Loading IndicTrans2...
IndicTrans2 failed: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/ai4bharat/indictrans2-en-indic-1B.
401 Client Error. (Request ID: Root=1-699c9161-6f2c2fa13f04efab66dc81e5;b401eb59-7f0f-41b9-9084-fd7cb77e4564)

Cannot access gated repo for url https://huggingface.co/ai4bharat/indictrans2-en-indic-1B/resolve/main/config.json.
Access to model ai4bharat/indictrans2-en-indic-1B is restricted. You must have access to it and be authenticated to access it. Please log in.
⚠️ IndicTrans2 failed. Attempting Google Translate fallback...
✓ Google Translate success!
✓ Hindi translation completed.
Text: अपना शरीर साफ़ करो, नहाओ, डियोडरेंट लगाओ, कपड़ा डालो तुम ये सब क्यों करते हो? क्योंकि आप हमेशा अपने ...


In [13]:
# ── Cell 13: Stage 5 — TTS Voice Synthesis ──────────────────────────────────
import os, re

# ┌─────────────────────────────────────────────────────────────────┐
# │  USE_XTTS = False  → gTTS  (fast, good Hindi, no voice clone)  │
# │  USE_XTTS = True   → XTTS  (voice clone, needs GPU / Colab)    │
# └─────────────────────────────────────────────────────────────────┘
USE_XTTS = True  # ← Change to True only on Colab GPU

TTS_RAW     = 'workspace/tts_raw.wav'
TTS_CLEAN   = 'workspace/tts_clean.wav'   # post-processed (clarity boost)
os.makedirs('workspace/tts_chunks', exist_ok=True)

# ── Hindi sentence splitter (respects । and . ! ?) ───────────────────────────
def split_hindi(text, max_chars=180):
    parts = re.split(r'(?<=[।.!?])\s*', text)
    chunks, cur = [], ''
    for p in parts:
        p = p.strip()
        if not p: continue
        if len(cur) + len(p) + 1 <= max_chars:
            cur = (cur + ' ' + p).strip()
        else:
            if cur: chunks.append(cur)
            cur = p
    if cur: chunks.append(cur)
    return chunks or [text]

def concat_wavs(chunk_paths, out_path, sample_rate=22050):
    """Concat wav files with 120ms silence between each."""
    !ffmpeg -y -f lavfi -i anullsrc=r={sample_rate}:cl=mono -t 0.12 workspace/sil.wav -loglevel warning
    parts = []
    for cp in chunk_paths:
        parts += [cp, 'workspace/sil.wav']
    with open('workspace/concat.txt', 'w') as f:
        for pp in parts: f.write(f"file '{os.path.abspath(pp)}'\n")
    !ffmpeg -y -f concat -safe 0 -i workspace/concat.txt -c copy {out_path} -loglevel warning

def enhance_audio(inp, out):
    """Post-process: normalize, gentle compression, de-ess, clarity EQ."""
    # acompressor: reduce dynamic range for consistent loudness
    # equalizer: boost presence (3kHz) for clarity, cut mud (250Hz)
    # loudnorm: broadcast standard volume
    !ffmpeg -y -i {inp} \
        -af "acompressor=threshold=0.05:ratio=3:attack=10:release=80:makeup=2,\
equalizer=f=250:t=o:w=200:g=-3,\
equalizer=f=3000:t=o:w=1000:g=3,\
loudnorm=I=-14:LRA=7:TP=-1" \
        -ar 44100 {out} -loglevel warning

# ── Option A: gTTS — fast Google Hindi TTS ───────────────────────────────────
if not USE_XTTS:
    from gtts import gTTS
    import subprocess
    print('🎙 gTTS — Google Hindi TTS (fast)...')
    chunks = split_hindi(HINDI_TEXT)
    print(f'Text: {HINDI_TEXT[:100]}...')
    print(f'Chunks: {len(chunks)}')
    chunk_paths = []
    for i, chunk in enumerate(chunks):
        mp3 = f'workspace/tts_chunks/chunk_{i:04d}.mp3'
        wav = f'workspace/tts_chunks/chunk_{i:04d}.wav'
        print(f'  [{i+1}/{len(chunks)}] {chunk[:70]}')
        gTTS(text=chunk, lang='hi', slow=False).save(mp3)
        # slow=False: natural speed; convert to 22050 wav
        !ffmpeg -y -i {mp3} -ar 22050 -ac 1 {wav} -loglevel warning
        chunk_paths.append(wav)
    concat_wavs(chunk_paths, TTS_RAW)
    print('✓ Raw TTS done — now enhancing clarity...')
    enhance_audio(TTS_RAW, TTS_CLEAN)
    print(f'✓ gTTS Hindi audio (enhanced): {TTS_CLEAN}')

# ── Option B: XTTS v2 — voice clone (GPU strongly recommended) ──────────────
else:
    from TTS.api import TTS
    os.environ['COQUI_TOS_AGREED'] = '1'
    print('🎙 XTTS v2 — Voice Clone (GPU recommended)...')
    tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2')
    if DEVICE == 'cuda': tts = tts.cuda()
    chunks = split_hindi(HINDI_TEXT, max_chars=150)  # Smaller = more stable
    print(f'Chunks: {len(chunks)}')
    chunk_paths = []
    for i, chunk in enumerate(chunks):
        p = f'workspace/tts_chunks/chunk_{i:04d}.wav'
        print(f'  [{i+1}/{len(chunks)}] {chunk[:70]}')
        tts.tts_to_file(
            text=chunk,
            speaker_wav=AUDIO_REF,
            language='hi',
            file_path=p,
            split_sentences=False,
        )
        chunk_paths.append(p)
    concat_wavs(chunk_paths, TTS_RAW)
    print('✓ Raw voice clone done — now enhancing clarity...')
    enhance_audio(TTS_RAW, TTS_CLEAN)
    print(f'✓ XTTS Hindi audio (enhanced): {TTS_CLEAN}')

# Share TTS_CLEAN as the output for next cells
TTS_FINAL_AUDIO = TTS_CLEAN
print(f'\n✅ TTS audio ready: {TTS_FINAL_AUDIO}')


/Users/vikashkumar/Desktop/Supernan/venv/lib/python3.11/site-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


🎙 XTTS v2 — Voice Clone (GPU recommended)...
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


GPT2InferenceModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Chunks: 2
  [1/2] अपना शरीर साफ़ करो, नहाओ, डियोडरेंट लगाओ, कपड़ा डालो तुम ये सब क्यों क
['अपना शरीर साफ़ करो, नहाओ, डियोडरेंट लगाओ, कपड़ा डालो तुम ये सब क्यों करते हो?']


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


 > Processing time: 11.624613046646118
 > Real-time factor: 1.3070220979774156
  [2/2] क्योंकि आप हमेशा अपने बच्चे के आसपास रहते हैं आपका बच्चा हमेशा आपके आस
['क्योंकि आप हमेशा अपने बच्चे के आसपास रहते हैं आपका बच्चा हमेशा आपके आसपास रहता है इसलिए अगर आप अपने बच्चे के शरीर को सूंघेंगे या उनकी सांसों को सूंघेंगे तो बच्चों को यह पसंद नहीं आएगा वे आपके पास नहीं आएंगे']
 > Processing time: 17.6355459690094
 > Real-time factor: 1.0943305321509784
zsh(68677) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68677) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(68677) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
zsh(68678) MallocStackLogging: process is not in a debuggable environment; unsetting 

In [14]:
# ── Cell 14: Stage 6 — Adjust Audio Speed to Match Clip ─────────────────────
import subprocess

TTS_ADJ  = 'workspace/tts_adjusted.wav'

# Use the enhanced (clarity-boosted) TTS audio from Cell 13
SOURCE_AUDIO = TTS_FINAL_AUDIO if 'TTS_FINAL_AUDIO' in dir() else TTS_RAW

def get_dur(path):
    r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration',
                        '-of','csv=p=0', path], capture_output=True, text=True)
    return float(r.stdout.strip())

def build_atempo(ratio):
    """Build chained atempo filters (each filter limited to 0.5-2.0x)."""
    filters, r = [], ratio
    while r < 0.5:  filters.append('atempo=0.5'); r /= 0.5
    while r > 2.0:  filters.append('atempo=2.0'); r /= 2.0
    filters.append(f'atempo={r:.6f}')
    return ','.join(filters)

tts_dur  = get_dur(SOURCE_AUDIO)
clip_dur = get_dur(CLIP)
ratio    = tts_dur / clip_dur

print(f'TTS duration : {tts_dur:.2f}s')
print(f'Clip duration: {clip_dur:.2f}s')
print(f'Speed ratio  : {ratio:.3f}x')

if 0.85 <= ratio <= 1.15:   # within ±15% — no stretching needed
    import shutil; shutil.copy(SOURCE_AUDIO, TTS_ADJ)
    print('✓ Speed close enough — copied without adjustment')
else:
    atempo = build_atempo(ratio)
    !ffmpeg -y -i {SOURCE_AUDIO} -af "{atempo}" -ar 44100 {TTS_ADJ} -loglevel warning
    print(f'✓ Speed adjusted ({ratio:.2f}x) → {clip_dur:.2f}s')


TTS duration : 23.20s
Clip duration: 20.00s
Speed ratio  : 1.160x
zsh(68695) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68695) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(68695) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[aist#0:0/pcm_s16le @ 0x73cc18180] Guessed Channel Layout: mono
✓ Speed adjusted (1.16x) → 20.00s


In [15]:
# ── Cell 15: Stage 7 — VideoReTalking Lip Sync ───────────────────────────────
import sys, os, subprocess
from IPython.display import Video

LIPSYNC = 'workspace/lipsync.mp4'
CHECKPOINT_DIR = 'models/VideoReTalking'
VRT_DIR = os.path.abspath('VideoReTalking')

# VideoReTalking needs its own third_part dirs in PYTHONPATH
vrt_env = {
    **os.environ,
    'PYTHONPATH': ':'.join([
        VRT_DIR,
        os.path.join(VRT_DIR, 'third_part'),
        os.path.join(VRT_DIR, 'third_part', 'face3d'),
        os.environ.get('PYTHONPATH', ''),
    ])
}

print('Running VideoReTalking (Stage 7)...')
result = subprocess.run(
    [sys.executable, 'VideoReTalking/inference.py',
     '--face',           CLIP,
     '--audio',          TTS_ADJ,
     '--outfile',        LIPSYNC,
     '--checkpoint_dir', CHECKPOINT_DIR,
    ],
    cwd='.',
    env=vrt_env,
)

if result.returncode != 0:
    print('⚠️  VRT failed, falling back to simple audio mux (no lip sync).')
    !ffmpeg -y -i {CLIP} -i {TTS_ADJ} -map 0:v -map 1:a \
        -c:v copy -c:a aac -shortest {LIPSYNC} -loglevel warning
    print('✓ Audio mux done (video = original clip + Hindi audio)')
else:
    print('✓ Lip-sync complete!')

print(f'Output: {LIPSYNC}')
Video(LIPSYNC, width=640)


Running VideoReTalking (Stage 7)...


Traceback (most recent call last):
  File "/Users/vikashkumar/Desktop/Supernan/VideoReTalking/inference.py", line 14, in <module>
    from third_part.face3d.extract_kp_videos import KeypointExtractor
  File "/Users/vikashkumar/Desktop/Supernan/VideoReTalking/third_part/face3d/extract_kp_videos.py", line 6, in <module>
    import face_alignment
ModuleNotFoundError: No module named 'face_alignment'


⚠️  VRT failed, falling back to simple audio mux (no lip sync).
zsh(68718) MallocStackLogging: process is not in a debuggable environment; unsetting MallocStackLoggingDirectory environment variable
zsh(68718) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
ffmpeg(68718) MallocStackLogging: could not tag MSL-related memory as no_footprint, so those pages will be included in process footprint - No such file or directory (2)
[aist#1:0/pcm_s16le @ 0x7bcc18900] Guessed Channel Layout: mono
✓ Audio mux done (video = original clip + Hindi audio)
Output: workspace/lipsync.mp4


In [ ]:
# ── Cell 16: Stage 8 — GFPGAN Face Enhancement ───────────────────────────────
SKIP_ENHANCEMENT = True  # Set to True to skip this slow stage and get output immediately
FINAL = "output/final_dubbed.mp4"
FRAMES_DIR = "workspace/frames_raw"
ENHANCED_DIR = "workspace/gfpgan_out"

import os, subprocess, torch

if SKIP_ENHANCEMENT:
    print("⏩ SKIP_ENHANCEMENT is True. Copying lip-sync video to final output...")
    get_ipython().system(f"cp {LIPSYNC} {FINAL}")
    print(f"✓ Final output (no enhancement): {FINAL}")
else:
    os.makedirs(FRAMES_DIR, exist_ok=True)
    print("Extracting frames...")
    get_ipython().system(f"ffmpeg -y -i {LIPSYNC} -qscale:v 1 -qmin 1 {FRAMES_DIR}/frame_%06d.png -loglevel warning")

    print("Running GFPGAN face restoration...")
    if torch.cuda.is_available():
        PROCESSOR = "cuda"
    elif torch.backends.mps.is_available():
        PROCESSOR = "mps"
    else:
        PROCESSOR = "cpu"
    print(f"Device detected: {PROCESSOR}")
    get_ipython().system(f"python GFPGAN/inference_gfpgan.py --ext png --device {PROCESSOR} -i {FRAMES_DIR} -o {ENHANCED_DIR} -v 1.4 -s 1 --bg_upsampler None")

    fps_r = subprocess.run(["ffprobe","-v","quiet","-select_streams","v:0",
                            "-show_entries","stream=r_frame_rate","-of","csv=p=0", LIPSYNC],
                           capture_output=True, text=True)
    FPS = fps_r.stdout.strip()

    RESTORED = f"{ENHANCED_DIR}/restored_imgs"
    print("Re-encoding final video...")
    get_ipython().system(f"ffmpeg -y -framerate {FPS} -pattern_type glob -i '{RESTORED}/*.png' -i {LIPSYNC} -map 0:v -map 1:a -c:v libx264 -crf 17 -preset slow -c:a aac -shortest {FINAL} -loglevel warning")

    final_dur = get_duration(FINAL)
    print(f"✓ Final output: {FINAL} ({final_dur:.2f}s)")

from IPython.display import Video
Video(FINAL, width=640)


In [ ]:
# ── Cell 17: Download Output ──────────────────────────────────────────────────
import os
try:
    from google.colab import files
    files.download(FINAL)
    print(f"✓ Download started for {FINAL}")
except ImportError:
    print(f"✓ Output saved to: {os.path.abspath(FINAL)}")
    print("   (Local machine detected, skipping browser download)")


---
## 📊 Pipeline Summary

| Stage | Tool | Duration |
|---|---|---|
| Clip Extract | ffmpeg | ~5s |
| Audio Extract | ffmpeg | ~2s |
| Transcription | Whisper medium | ~30s |
| Translation | IndicTrans2 | ~60s |
| Voice Clone | Coqui XTTS v2 | ~3-5 min |
| Speed Adjust | ffmpeg atempo | ~5s |
| Lip Sync | VideoReTalking | ~10-20 min |
| Face Enhance | GFPGAN v1.4 | ~2-3 min |

**Total on free Colab T4: ~15-30 minutes for a 15-second clip**

**Cost: ₹0** ✅